In [ ]:
%%capture
%pip install pandas psycopg2-binary ipython-sql # You may need to click on "Restart this kernel" to use these packages

In [ ]:
import sqlalchemy
import pandas
from bindings import bindings
my_binding = bindings.find(SERVICEBINDING, "enter-database-binding")
engine = sqlalchemy.create_engine(sqlalchemy.engine.url.URL.create(drivername='postgresql',username=my_binding.get('username'),host=my_binding.get('host'),port=my_binding.get('port'),database=my_binding.get('database'),password=my_binding.get('password')))
url = engine.url.render_as_string(hide_password=False)
%load_ext sql

In [ ]:
%%sql $url
CREATE OR REPLACE FUNCTION llm_translate(input text)
RETURNS TEXT
AS $$
DECLARE translation text;
BEGIN
    SELECT pgml.transform(
        task => '{
            "task": "translation",
            "model": "Helsinki-NLP/opus-mt-en-fr",
            "torch_dtype": "bfloat16"
        }'::JSONB,
        inputs => ARRAY [
            input
        ]
    )
    INTO translation;
    RETURN translation;
END;
$$
LANGUAGE 'plpgsql';
select llm_translate('Do you speak French?')

In [ ]:
%%sql $url
CREATE OR REPLACE FUNCTION nlp_translate(input text)
RETURNS TEXT
AS $$
    from translate import Translator
    translator = Translator(from_lang='en', to_lang='fr')
    translation = translator.translate(input)
    return translation
$$
LANGUAGE 'plpython3u';
select nlp_translate('Do you speak French?')

In [ ]:
%%sql $url
SELECT tweet as sentence_eng,
        nlp_translate(tweet) as sentence_french
FROM public.tweets
LIMIT 40;

In [ ]:
%%sql $url
SELECT tweet as sentence_eng,
        nlp_translate(tweet) as sentence_french_mlt,
        llm_translate(tweet) as sentence_french_llm
FROM public.tweets
LIMIT 5;